# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# %matplotlib widget

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import hist
import mplhep as hep


from rich import print
from matplotlib.colors import LogNorm
from tpvalidator.analysis.histograms import make_regaxis, build_histogram


from IPython.display import Markdown

# Data

In [ ]:
import tpvalidator.datacatalogue as dctl

dataset_names = ['ar39']
# datasets = dctl.load('data/vd/1x8x14/old_detsim', dataset_names)
datasets = dctl.load('data/vd/1x8x14/3sig', dataset_names)


In [ ]:
ar39_ws = datasets['ar39']

In [ ]:
ar39_coll_tps = ar39_ws.tps.query('bt_is_signal == True & readout_plane_id == 2')


In [ ]:

ar39_coll_tps.bt_primary_x.hist(bins=100, weights=ar39_coll_tps.adc_integral)

In [ ]:
from tpvalidator.analysis.histograms import build_histogram, make_regaxis

bt_x_axies = make_regaxis(ar39_coll_tps.query('adc_peak > 45 & samples_over_threshold >= 9'), 'bt_primary_x', 10)

h = build_histogram(ar39_coll_tps, [bt_x_axies], weight='adc_integral')

In [ ]:
import mplhep as hep

In [ ]:
display(h)

## Fitting the `adc_integral` counts as a function of drift depth.

In [ ]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import LeastSquares

# Exponential model: N0 * exp(-x / tau)
def model(x, N0, x0, tau):
    return N0 * np.exp(-(x-x0) / tau)

bin_centers = -h.axes[0].centers
y = h.values()
yerr = np.sqrt(h.variances())

cost = LeastSquares(bin_centers, y, yerr, model)
m = Minuit(cost, N0=y.max(), x0=0, tau=100.0)
m.migrad()
m.hesse()

# print(m.values)   # fitted parameters
# print(m.errors)   # parameter uncertainties
# print(m.fmin)      # fit quality (chi2, ndof, valid, etc.)